# LeetCode 756 — Pyramid Transition Matrix bằng Python

Link đề: https://leetcode.com/problems/pyramid-transition-matrix/ · Độ khó: **Medium** · Chủ đề: Backtracking, Bit Manipulation, Depth-First Search, Breadth-First Search, Hash Table

Skeleton LeetCode:

```python
class Solution:
    def pyramidTransition(self, bottom: str, allowed: List[str]) -> bool:
        ...
```

Ghi chú: ví dụ 3 ở mục 2 và các ca viết tay ở mục 12 là **tự dựng** (đáp án đã tính tay và kiểm chứng bằng code); nếu cần khớp từng chữ với đề gốc, hãy đối chiếu lại. Đây là bài **quay lui (backtracking)** trên một không gian nhỏ (`bottom` tối đa 6 ký tự, 6 chữ cái), nên điểm mấu chốt là **biểu diễn trạng thái** (cả một hàng là một chuỗi) và **nhớ** những hàng đã thử.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: xếp một kim tự tháp từ hàng đáy `bottom`; mỗi hàng ngắn hơn hàng dưới **một khối** và đặt chính giữa; mỗi **tam giác** (hai khối ở dưới, một khối ở trên) phải nằm trong danh sách mẫu cho phép `allowed`. Hỏi có xây được tới **đỉnh** không.
2. Nắm các công cụ Python: tách các cặp kề nhau (`zip(s, s[1:])`, `itertools.pairwise`), `dict` với khóa là **tuple** `(a, b)`, `itertools.product(*các_danh_sách)` (tích Descartes), `any(...)` ngắn mạch, ghi nhớ kết quả bằng `functools.lru_cache` hoặc `set`, và **bitmask** để lưu một tập chữ cái trong một số nguyên.
3. Đi từ **thử mọi cách dựng hàng kế tiếp (không nhớ)** → **nhớ những hàng đã thử** (memo) → **dựng từng ký tự của hàng kế tiếp bằng bitmask + cắt nhánh** → **BFS theo từng tầng với tập các hàng**.
4. Hiểu mấu chốt: **trạng thái** của bài toán chỉ là **hàng hiện tại** (một chuỗi) — mọi cách đi tới cùng một hàng đều có tương lai y hệt, nên mỗi hàng chỉ cần giải **một lần** (memo). Mỗi cặp kề nhau cho một **danh sách** ký tự đỉnh có thể, và hàng kế tiếp là **tích Descartes** các danh sách đó.
5. Tránh các lỗi kinh điển: coi tam giác **không phân biệt trái–phải**, tham lam chọn **ký tự đỉnh đầu tiên**, **bỏ qua** cặp không có mẫu thay vì trả `False`, dùng bộ nhớ đệm **toàn cục** khi bảng `allowed` thay đổi giữa các lần gọi, và sai chỉ số khi tách cặp kề.

## 2. Đề bài & ví dụ

Bạn đang xếp các khối để tạo thành một **kim tự tháp**. Mỗi khối có một màu, biểu diễn bằng **một chữ cái**. Mỗi hàng có **ít hơn một khối** so với hàng dưới nó và được **đặt chính giữa** phía trên.

Để kim tự tháp đẹp, chỉ cho phép một số **mẫu tam giác**. Một mẫu tam giác gồm **một khối** đặt trên **hai khối**. Các mẫu cho trước dưới dạng danh sách chuỗi 3 chữ cái `allowed`: **hai ký tự đầu** là khối đáy **trái** và **phải**, ký tự **thứ ba** là khối ở **đỉnh**.

- Ví dụ `"ABC"` là mẫu có khối `C` đặt trên khối `A` (trái) và `B` (phải). Khác với `"BAC"` (khối `B` bên trái, `A` bên phải).

Bạn bắt đầu với hàng đáy `bottom` (một chuỗi) mà **bắt buộc** dùng làm đáy. Với `bottom` và `allowed`, trả về `true` nếu có thể xây kim tự tháp **lên tới đỉnh** sao cho **mọi tam giác** trong tháp đều thuộc `allowed`, ngược lại `false`.

**Ví dụ 1**

- Input: `bottom = "BCD"`, `allowed = ["BCC","CDE","CEA","FFF"]`
- Output: `true`
- Giải thích: từ đáy (tầng 3), xây `"CE"` ở tầng 2 (mẫu `BCC` và `CDE`), rồi xây `"A"` ở tầng 1 (mẫu `CEA`). Ba tam giác đều hợp lệ.

**Ví dụ 2**

- Input: `bottom = "AAAA"`, `allowed = ["AAB","AAC","BCD","BBE","DEF"]`
- Output: `false`
- Giải thích: có nhiều cách xây tầng 3, nhưng thử hết đều bị kẹt trước khi tới đỉnh.

**Ví dụ 3 (tự dựng, đối chiếu lại đề gốc nếu cần)**

- Input: `bottom = "AB"`, `allowed = ["ABC"]` → `true` (chỉ cần một tam giác `ABC`).
- Input: `bottom = "AB"`, `allowed = ["BAC"]` → `false` (mẫu `BAC` cần `B` bên trái, `A` bên phải — ngược với đáy).

**Ràng buộc:**

- `2 <= bottom.length <= 6`
- `0 <= allowed.length <= 216`
- `allowed[i].length == 3`
- Mọi chữ cái thuộc tập `{'A','B','C','D','E','F'}`.
- Các giá trị của `allowed` là **duy nhất**.

In [1]:
from typing import List
from collections import deque
from functools import lru_cache
from itertools import product, pairwise
import random
import sys

examples = [
    ("BCD", ["BCC", "CDE", "CEA", "FFF"], True),
    ("AAAA", ["AAB", "AAC", "BCD", "BBE", "DEF"], False),
    ("AB", ["ABC"], True),
    ("AB", ["BAC"], False),
]
for bottom, allowed, expected in examples:
    print(f"bottom={bottom!r} allowed={allowed} -> kỳ vọng {expected}")

bottom='BCD' allowed=['BCC', 'CDE', 'CEA', 'FFF'] -> kỳ vọng True
bottom='AAAA' allowed=['AAB', 'AAC', 'BCD', 'BBE', 'DEF'] -> kỳ vọng False
bottom='AB' allowed=['ABC'] -> kỳ vọng True
bottom='AB' allowed=['BAC'] -> kỳ vọng False


## 3. Kiến thức nền

### 3.1. Tách các cặp kề nhau của một hàng

Mỗi tam giác ở hàng trên dựa trên **hai khối kề nhau** của hàng dưới. Một chuỗi `s` độ dài `m` có `m - 1` cặp kề nhau: `(s[0], s[1])`, `(s[1], s[2])`, ... Hai cách viết: `zip(s, s[1:])` và `itertools.pairwise(s)` (Python 3.10+). Cẩn thận: dùng `range(len(s))` rồi `s[i:i+2]` sẽ có cặp cuối **chỉ một ký tự** — lỗi hay gặp ở mục 13.

In [2]:
s = "BCD"
print(list(zip(s, s[1:])))
print(list(pairwise(s)))
print([s[i:i + 2] for i in range(len(s))])     # cặp cuối chỉ có 1 ký tự: 'D'
print("số cặp của hàng độ dài", len(s), "là", len(s) - 1)

[('B', 'C'), ('C', 'D')]
[('B', 'C'), ('C', 'D')]
['BC', 'CD', 'D']
số cặp của hàng độ dài 3 là 2


### 3.2. `dict` với khóa là tuple; `setdefault` và `get`

Ta cần tra "hai khối đáy `(a, b)` → các khối đỉnh cho phép". Khóa là tuple `(a, b)` (phân biệt thứ tự trái–phải: `('A','B')` khác `('B','A')`). Giá trị là **list** các ký tự đỉnh. `d.setdefault(key, [])` lấy list của `key`, nếu chưa có thì tạo list rỗng; `d.get(key, [])` đọc mà không tạo khóa.

In [3]:
allowed = ["BCC", "CDE", "CEA", "FFF", "BCD"]
d = {}
for a, b, c in allowed:                 # tách chuỗi 3 ký tự thành 3 biến
    d.setdefault((a, b), []).append(c)
print(d)
print("('B','C') ->", d.get(("B", "C"), []))
print("('C','B') ->", d.get(("C", "B"), []), "(thứ tự trái-phải quan trọng)")

{('B', 'C'): ['C', 'D'], ('C', 'D'): ['E'], ('C', 'E'): ['A'], ('F', 'F'): ['F']}
('B','C') -> ['C', 'D']
('C','B') -> [] (thứ tự trái-phải quan trọng)


### 3.3. `itertools.product(*các_danh_sách)` — tích Descartes

`product(L1, L2, L3)` sinh mọi bộ `(x1, x2, x3)` với `x1 ∈ L1`, `x2 ∈ L2`, `x3 ∈ L3` (theo thứ tự từ điển, `L1` đổi chậm nhất). Dấu `*` trước một list **giải nén** nó thành các đối số. Với bài này: mỗi cặp kề nhau cho một danh sách đỉnh có thể; **mọi hàng kế tiếp** là một phần tử của tích Descartes các danh sách đó; ghép bộ ký tự thành chuỗi bằng `"".join(...)`.

In [4]:
lists = [["C"], ["D", "E"], ["A", "B"]]
for combo in product(*lists):
    print(combo, "->", "".join(combo))
print("số hàng kế tiếp =", 1 * 2 * 2)
print(list(product(*[])))                # tích rỗng: đúng MỘT bộ rỗng

('C', 'D', 'A') -> CDA
('C', 'D', 'B') -> CDB
('C', 'E', 'A') -> CEA
('C', 'E', 'B') -> CEB
số hàng kế tiếp = 4
[()]


### 3.4. `any(...)` và ngắn mạch

`any(biểu_thức for x in ...)` trả `True` ngay khi gặp phần tử đúng đầu tiên và **không** tính tiếp phần còn lại (bộ sinh là **lười**). Rất hợp cho quay lui: "có **một** hàng kế tiếp nào đó xây được không?".

In [5]:
def check(x):
    print("  thử", x)
    return x >= 3

print("any:", any(check(x) for x in [1, 2, 3, 4, 5]))     # dừng sau khi thử 3

  thử 1
  thử 2
  thử 3
any: True


### 3.5. Ghi nhớ kết quả: `lru_cache` và `set`

Khi một hàm đệ quy được gọi nhiều lần với **cùng đối số**, ta lưu kết quả để khỏi tính lại (**memoization**). Hai cách: dán `@lru_cache(None)` lên hàm (đối số phải **hashable**, chuỗi thì được), hoặc tự giữ một `set` các trạng thái **đã biết là thất bại**. Hãy đếm số lần thân hàm thật sự chạy để thấy hiệu quả.

In [6]:
calls = 0

def fib_plain(n):
    global calls
    calls += 1
    return n if n < 2 else fib_plain(n - 1) + fib_plain(n - 2)

calls = 0; fib_plain(15); print("không memo: số lần gọi =", calls)

calls2 = 0
@lru_cache(None)
def fib_memo(n):
    global calls2
    calls2 += 1
    return n if n < 2 else fib_memo(n - 1) + fib_memo(n - 2)

fib_memo(15); print("có memo   : số lần chạy thân hàm =", calls2)

không memo: số lần gọi = 1973
có memo   : số lần chạy thân hàm = 16


### 3.6. Bitmask: lưu một **tập chữ cái** trong một số nguyên

Có 6 chữ cái `A..F`, ta mã hóa mỗi chữ thành một bit: `'A' -> bit 0`, `'B' -> bit 1`, ... Một tập chữ cái là số nguyên `mask`: bật bit `i` bằng `mask |= 1 << i`; kiểm tra bit `i` bằng `mask >> i & 1`. `ord(c) - 65` đổi chữ in hoa thành số `0..25` (`ord('A') = 65`); `chr(65 + i)` đổi ngược.

In [7]:
mask = 0
for c in "CE":
    mask |= 1 << (ord(c) - 65)
print("mask của {C, E} =", mask, "=", bin(mask))
print("những chữ trong mask:", [chr(65 + i) for i in range(6) if mask >> i & 1])
print("có 'E' không?", bool(mask >> (ord("E") - 65) & 1), "| có 'D' không?", bool(mask >> (ord("D") - 65) & 1))

mask của {C, E} = 20 = 0b10100
những chữ trong mask: ['C', 'E']
có 'E' không? True | có 'D' không? False


### 3.7. Vẽ kim tự tháp bằng chữ

Hàm `show(rows)` nhận danh sách hàng **từ đáy lên đỉnh** và in tháp (đỉnh ở trên), mỗi hàng được căn giữa bằng `str.center`.

In [8]:
def show(rows):
    width = len(rows[0]) * 2 - 1
    for row in reversed(rows):
        print(" ".join(row).center(width))

show(["BCD", "CE", "A"])

  A  
 C E 
B C D


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1: `bottom = "BCD"`, `allowed = ["BCC","CDE","CEA","FFF"]`.

Tầng 2 có 2 khối, mỗi khối nằm trên một cặp kề của đáy:

- Cặp `(B, C)`: mẫu bắt đầu bằng `BC` là `BCC` → khối đỉnh **chỉ có thể là `C`**.
- Cặp `(C, D)`: mẫu `CDE` → khối đỉnh **chỉ có thể là `E`**.

Tầng 2 vì vậy chỉ có thể là `"CE"`. Tầng 1: cặp `(C, E)`: mẫu `CEA` → đỉnh `A`. Xong, tháp xây được.

Tổng quát: mỗi cặp có **một danh sách** ký tự đỉnh hợp lệ (có thể rỗng, một, hoặc nhiều). Nếu một cặp có danh sách **rỗng** thì **không** xây nổi hàng trên → thất bại ngay. Nếu nhiều lựa chọn, ta phải **thử** (quay lui) vì không biết lựa chọn nào dẫn tới đỉnh.

Ta viết hai công cụ dùng xuyên suốt:

1. `build_table(allowed)` dựng bảng `(a, b) → [các đỉnh]`.
2. `is_valid_pyramid(rows, allowed)`: kiểm tra một tháp **cụ thể** (từ đáy lên đỉnh) có hợp lệ không — để kiểm chứng.

In [9]:
def build_table(allowed):
    d = {}
    for a, b, c in allowed:
        d.setdefault((a, b), []).append(c)
    return d

def is_valid_pyramid(rows, allowed):
    """rows: các hàng từ đáy lên đỉnh. Hợp lệ nếu mỗi hàng ngắn hơn 1, đỉnh dài 1, và mọi tam giác thuộc allowed."""
    S = set(allowed)
    if len(rows[-1]) != 1:
        return False
    for lower, upper in zip(rows, rows[1:]):
        if len(upper) != len(lower) - 1:
            return False
        for c in range(len(upper)):
            if lower[c] + lower[c + 1] + upper[c] not in S:
                return False
    return True

d = build_table(["BCC", "CDE", "CEA", "FFF"])
print(d)
print(is_valid_pyramid(["BCD", "CE", "A"], ["BCC", "CDE", "CEA", "FFF"]))      # True
print(is_valid_pyramid(["BCD", "CC", "A"], ["BCC", "CDE", "CEA", "FFF"]))      # False: CD -> C không có mẫu

{('B', 'C'): ['C'], ('C', 'D'): ['E'], ('C', 'E'): ['A'], ('F', 'F'): ['F']}
True
False


### Một "trọng tài" vét cạn hoàn toàn độc lập

Với bảng chữ cái **nhỏ** và tháp **thấp**, ta có thể **thử mọi cách điền mọi ô** phía trên đáy rồi kiểm tra từng tam giác — không dùng bảng tra, không dùng đệ quy theo hàng. Số ô phía trên đáy là `n(n-1)/2`; với bảng chữ cái kích thước `s` có `s^(n(n-1)/2)` cách điền. Với `s = 2..3`, `n ≤ 5` còn chấp nhận được.

In [10]:
def oracle(bottom, allowed, letters):
    """Trọng tài: thử mọi cách điền các ô phía trên đáy (chỉ dùng chữ trong `letters`)."""
    n = len(bottom)
    S = set(allowed)
    total = n * (n - 1) // 2
    for assign in product(letters, repeat=total):
        rows = [bottom]
        idx = 0
        for length in range(n - 1, 0, -1):
            rows.append("".join(assign[idx:idx + length]))
            idx += length
        if all(rows[r][c] + rows[r][c + 1] + rows[r + 1][c] in S
               for r in range(n - 1) for c in range(len(rows[r + 1]))):
            return True
    return False

for bottom, allowed, expected in examples:
    letters = sorted(set("".join(allowed)) | set(bottom))
    if len(letters) ** (len(bottom) * (len(bottom) - 1) // 2) <= 200000:
        assert oracle(bottom, allowed, letters) == expected, (bottom, allowed)
        print(f"{bottom!r}: oracle -> {oracle(bottom, allowed, letters)}")

'BCD': oracle -> True
'AAAA': oracle -> False
'AB': oracle -> True
'AB': oracle -> False


(Ví dụ 1 có 6 chữ cái và đáy 3 → `6^3 = 216` cách; ví dụ 2 dùng `AAAA` với các chữ `A..F` → `6^6 = 46656` cách — cả hai đều vừa với giới hạn `200000`, nên đều được kiểm.)

## 5. Cách 1 — Brute Force: thử mọi cách dựng hàng kế tiếp (không nhớ)

**Ý tưởng.** Hàm `can(row)`:

1. Nếu `len(row) == 1` → đã tới đỉnh, trả `True`.
2. Với mỗi cặp kề nhau `(a, b)` lấy danh sách đỉnh `d.get((a, b), [])`. Nếu có cặp **không** có mẫu → hàng kế tiếp không dựng được (trả `False`).
3. **Liệt kê mọi hàng kế tiếp** (tích Descartes) rồi hỏi `can(next_row)` cho từng hàng; có **một** hàng thành công là đủ (`any`).

Ta viết hàm `expand_all(row, d)` trả list mọi hàng kế tiếp (dựng dần: bắt đầu `[""]`, mỗi cặp nối thêm từng ký tự đỉnh).

In [11]:
def expand_all(row, d):
    """Mọi hàng kế tiếp (độ dài len(row)-1); [] nếu có cặp không có mẫu."""
    rows = [""]
    for a, b in zip(row, row[1:]):
        tops = d.get((a, b), [])
        rows = [r + t for r in rows for t in tops]
    return rows

def brute_force(bottom, allowed, counter=None, cap=None):
    d = build_table(allowed)

    def can(row):
        if counter is not None:
            counter[0] += 1
            if cap is not None and counter[0] > cap:
                raise RuntimeError("quá nhiều lần gọi")
        if len(row) == 1:
            return True
        return any(can(nxt) for nxt in expand_all(row, d))

    return can(bottom)

for bottom, allowed, expected in examples:
    got = brute_force(bottom, allowed)
    assert got == expected, (bottom, allowed, got, expected)
    print(f"{bottom!r} -> {got}")

'BCD' -> True
'AAAA' -> False
'AB' -> True
'AB' -> False


**Độ phức tạp.** Số hàng kế tiếp có thể tới `(số đỉnh mỗi cặp)^(m-1)` (tối đa `6^(m-1)`), rồi mỗi hàng lại sinh tiếp... → **Time cỡ hàm mũ theo `n`** (không nhớ nên lặp lại rất nhiều). **Space O(số hàng kế tiếp)** cho `expand_all` cộng độ sâu đệ quy ≤ `n`. Với `n ≤ 6` vẫn chạy được ở nhiều ca nhưng có thể chậm.

**Nút thắt.** Nhiều **đường đi khác nhau** dẫn tới **cùng một hàng**, và ta giải đi giải lại cùng hàng đó. Hãy đếm xem.

Mỗi lần `can(row)` được gọi cho một hàng, và **cùng một hàng có thể xuất hiện lại** từ những hàng cha khác nhau (hoặc cùng một cha qua nhiều tổ hợp). Số hàng **khác nhau** có thể có thì hữu hạn nhỏ (tối đa `6^5 + 6^4 + ... + 6 + 1` với 6 chữ cái), nhưng brute force giải lại **mỗi lần** xuất hiện. Lời giải: **mỗi hàng chỉ giải một lần** — ta sẽ đo ngay ở mục sau.

## 6. Ý tưởng thuật toán / bước đột phá

**Trạng thái = hàng hiện tại.** Việc hàng này có xây tiếp lên tới đỉnh được hay không **chỉ phụ thuộc vào chính chuỗi hàng đó** (và bảng `allowed` cố định), không phụ thuộc cách ta đã tới đó. Vậy ta có thể:

- Tính `can(row)` **một lần** rồi nhớ (memo).
- Số hàng khác nhau tối đa: độ dài `m` có `≤ 6^m` hàng, tổng nhỏ cho `m ≤ 6`.

Ngoài ra ta tối ưu **cắt nhánh sớm**: ngay khi một cặp không có mẫu thì trả `False` mà **không** cần liệt kê gì.

Hãy so số lần thân hàm chạy **trước và sau memo** trên một ca chọn ra từ 1500 ca ngẫu nhiên (cùng seed).

In [12]:
def memo_product(bottom, allowed, counter=None):
    d = build_table(allowed)

    @lru_cache(None)
    def dfs(s):
        if counter is not None:
            counter[0] += 1
        if len(s) == 1:
            return True
        tops = []
        for a, b in pairwise(s):
            cs = d.get((a, b))
            if not cs:
                return False                    # có cặp không có mẫu: cắt nhánh ngay
            tops.append(cs)
        return any(dfs("".join(nxt)) for nxt in product(*tops))

    return dfs(bottom)

# Tìm (có seed) ca đáy 6 chữ trên bảng A..D, kết quả False, mà memo tiết kiệm nhiều nhất so với brute force
allp4 = [a + b + c for a in "ABCD" for b in "ABCD" for c in "ABCD"]
rng = random.Random(756)
best = None
for _ in range(1500):
    allowed = rng.sample(allp4, rng.randint(16, 32))
    bottom = "".join(rng.choice("ABCD") for _ in range(6))
    c1, c2 = [0], [0]
    try:
        r1 = brute_force(bottom, allowed, c1, cap=60000)
    except RuntimeError:
        continue
    r2 = memo_product(bottom, allowed, c2)
    assert r1 == r2
    if not r1 and (best is None or c1[0] - c2[0] > best[0]):
        best = (c1[0] - c2[0], c1[0], c2[0], bottom, len(allowed))
print(f"ca tiết kiệm nhiều nhất (đáy 6 chữ, kết quả False, {best[4]} mẫu, bottom = {best[3]}):")
print(f"  brute force: {best[1]} lần gọi | memo: {best[2]} lần chạy thân hàm")
assert best[2] < best[1]
print("(với đề thật, các ca khó hơn còn lặp nhiều hơn nữa; memo không bao giờ chạy nhiều hơn brute force)")

ca tiết kiệm nhiều nhất (đáy 6 chữ, kết quả False, 22 mẫu, bottom = DBCCBC):
  brute force: 70 lần gọi | memo: 29 lần chạy thân hàm
(với đề thật, các ca khó hơn còn lặp nhiều hơn nữa; memo không bao giờ chạy nhiều hơn brute force)


## 7. Các cách tiếp theo

4 cách tổng cộng (cách 1 đã ở mục 5). Gọi `n = len(bottom)`, `s = 6` (số chữ cái):

| # | Cách | Ý tưởng | Ghi chú độ phức tạp |
|---|---|---|---|
| 1 | Brute force không nhớ | thử mọi hàng kế tiếp | hàm mũ, lặp nhiều |
| 2 | Memo + `product` | nhớ mỗi hàng một lần | ≤ `Σ 6^m` hàng; mỗi hàng ≤ `6^(m-1)` hàng con |
| 3 | Dựng từng ký tự + bitmask + `failed` | cắt nhánh theo từng ký tự | cùng bậc nhưng cắt sớm |
| 4 | BFS theo tầng với tập hàng | gom mọi hàng cùng độ dài | tốn khi quá nhiều hàng |

### 7.1. Cách 2 — Memo + `product` (DFS theo hàng)

**Giải thích từng đoạn.**

1. `d`: bảng `(a, b) → [đỉnh]` (mục 4).
2. `dfs(s)` có `@lru_cache(None)`: nhớ kết quả cho mỗi chuỗi `s`.
3. Cơ sở: `len(s) == 1` → `True`.
4. Với mỗi cặp kề nhau lấy `cs = d.get((a, b))`; nếu rỗng/không có → `False`; nếu không thì thêm vào `tops`.
5. `product(*tops)` sinh mọi hàng kế tiếp; `any(dfs("".join(nxt)) ...)` thử lần lượt và dừng ở hàng đầu tiên thành công.

In [13]:
for bottom, allowed, expected in examples:
    got = memo_product(bottom, allowed)
    assert got == expected, (bottom, allowed, got, expected)
    print(f"{bottom!r} -> {got}")

'BCD' -> True
'AAAA' -> False
'AB' -> True
'AB' -> False


In [14]:
# Code nộp LeetCode — Cách 2: memo + product
from functools import cache
from itertools import pairwise, product


class Solution:
    def pyramidTransition(self, bottom: str, allowed: List[str]) -> bool:
        d = {}
        for a, b, c in allowed:
            d.setdefault((a, b), []).append(c)

        @cache
        def dfs(s: str) -> bool:
            if len(s) == 1:
                return True
            tops = []
            for a, b in pairwise(s):
                cs = d.get((a, b))
                if not cs:
                    return False
                tops.append(cs)
            return any(dfs("".join(nxt)) for nxt in product(*tops))

        return dfs(bottom)

for bottom, allowed, expected in examples:
    assert Solution().pyramidTransition(bottom, allowed) == expected
print("Cách 2 qua ví dụ đề")

Cách 2 qua ví dụ đề


**Độ phức tạp.** Số hàng khác nhau độ dài `m` tối đa `6^m`; mỗi hàng được giải **một lần**, mỗi lần tốn tối đa `6^(m-1)` để liệt kê hàng con (tệ nhất khoảng `10^7` thao tác tạo chuỗi ở `m = 5`, nhưng thực tế các cặp hiếm khi đều có nhiều đỉnh nên nhỏ hơn rất nhiều). **Space O(Σ 6^m)** cho bộ nhớ đệm (tối đa vài chục nghìn chuỗi ngắn).

### 7.2. Cách 3 — Dựng từng ký tự của hàng kế tiếp (bitmask) + tập thất bại

**Ý tưởng.** Thay vì liệt kê **cả hàng** kế tiếp rồi mới đệ quy, ta dựng nó **từng ký tự một** bằng DFS với hai tham số: `s` (hàng hiện tại) và `t` (hàng kế tiếp đang dựng dở):

- Nếu `len(s) == 1` → `True`.
- Nếu `len(t) + 1 == len(s)` → hàng kế tiếp đã đủ → đệ quy `dfs(t, "")`.
- Ngược lại, ký tự tiếp theo của `t` nằm trên cặp `(s[len(t)], s[len(t)+1])`; thử mọi ký tự đỉnh cho phép của cặp đó.

**Bitmask:** bảng `d[a][b]` là một số nguyên 6 bit (mỗi bit một ký tự đỉnh cho phép) — duyệt `i` từ `0..5` và thử ký tự `chr(65 + i)` nếu bit bật. Bảng là mảng `7×7` (chỉ số `ord(c) - 65`).

**Tập thất bại `failed`:** lưu cặp `(s, t)` đã thử mà thất bại; gặp lại thì trả `False` ngay. (Thành công thì ta đã thoát toàn bộ đệ quy với `True`, nên không cần lưu.)

Ưu điểm: cắt nhánh **sớm** — nếu cặp thứ 3 không có mẫu, ta bỏ ngay mà không cần xây hết các ký tự còn lại cho các cặp phía sau.

In [15]:
def charwise_bitmask(bottom, allowed):
    d = [[0] * 7 for _ in range(7)]
    for a, b, c in allowed:
        d[ord(a) - 65][ord(b) - 65] |= 1 << (ord(c) - 65)
    failed = set()

    def dfs(s, t):
        if len(s) == 1:
            return True
        if len(t) + 1 == len(s):
            return dfs(t, "")
        key = (s, t)
        if key in failed:
            return False
        mask = d[ord(s[len(t)]) - 65][ord(s[len(t) + 1]) - 65]
        for i in range(6):
            if mask >> i & 1 and dfs(s, t + chr(65 + i)):
                return True
        failed.add(key)
        return False

    return dfs(bottom, "")

for bottom, allowed, expected in examples:
    got = charwise_bitmask(bottom, allowed)
    assert got == expected, (bottom, allowed, got, expected)
    print(f"{bottom!r} -> {got}")

'BCD' -> True
'AAAA' -> False
'AB' -> True
'AB' -> False


In [16]:
# Code nộp LeetCode — Cách 3: từng ký tự + bitmask + failed
class Solution:
    def pyramidTransition(self, bottom: str, allowed: List[str]) -> bool:
        d = [[0] * 7 for _ in range(7)]
        for a, b, c in allowed:
            d[ord(a) - 65][ord(b) - 65] |= 1 << (ord(c) - 65)
        failed = set()

        def dfs(s: str, t: str) -> bool:
            if len(s) == 1:
                return True
            if len(t) + 1 == len(s):
                return dfs(t, "")
            key = (s, t)
            if key in failed:
                return False
            mask = d[ord(s[len(t)]) - 65][ord(s[len(t) + 1]) - 65]
            for i in range(6):
                if mask >> i & 1 and dfs(s, t + chr(65 + i)):
                    return True
            failed.add(key)
            return False

        return dfs(bottom, "")

for bottom, allowed, expected in examples:
    assert Solution().pyramidTransition(bottom, allowed) == expected
print("Cách 3 qua ví dụ đề")

Cách 3 qua ví dụ đề


**Độ phức tạp.** Cùng bậc với cách 2 trong tệ nhất nhưng thường **nhanh hơn nhiều** vì cắt nhánh ngay từ ký tự đầu của hàng kế tiếp. Độ sâu đệ quy ≤ tổng số ký tự phải điền (≤ `5 + 4 + 3 + 2 + 1 = 15` lần đặt ký tự, cộng chuyển hàng) — rất nông. **Space** `O(số trạng thái (s, t) thất bại)` + độ sâu đệ quy.

### 7.3. Cách 4 — BFS theo từng tầng với tập các hàng

**Ý tưởng.** Thay vì đi sâu một đường, ta **gom mọi hàng có thể có ở từng tầng**: `level` = tập các hàng độ dài `m` mà ta có thể có. Từ mỗi hàng trong `level`, sinh mọi hàng kế tiếp (nếu mọi cặp đều có mẫu) và thêm vào tập `nxt` (tập tự **khử trùng**: đây chính là "memo" của BFS). Lặp cho tới khi độ dài còn `1`; nếu tập `nxt` rỗng ở bước nào đó thì thất bại.

BFS trả lời thẳng "tầng đỉnh có tới được không" mà không cần đệ quy. Nhược điểm: nó **sinh hết** mọi hàng có thể có ở mỗi tầng (không dừng sớm khi tìm thấy đường), nên với `allowed` rất dày (216 mẫu) và đáy dài 6 sẽ chậm hơn các cách DFS.

In [17]:
def bfs_levels(bottom, allowed):
    d = build_table(allowed)
    level = {bottom}
    while len(next(iter(level))) > 1:
        nxt = set()
        for row in level:
            tops = []
            for a, b in pairwise(row):
                cs = d.get((a, b))
                if not cs:
                    break
                tops.append(cs)
            else:
                for combo in product(*tops):
                    nxt.add("".join(combo))
        if not nxt:
            return False
        level = nxt
    return True

for bottom, allowed, expected in examples:
    got = bfs_levels(bottom, allowed)
    assert got == expected, (bottom, allowed, got, expected)
    print(f"{bottom!r} -> {got}")

'BCD' -> True
'AAAA' -> False
'AB' -> True
'AB' -> False


In [18]:
# Code nộp LeetCode — Cách 4: BFS theo tầng
class Solution:
    def pyramidTransition(self, bottom: str, allowed: List[str]) -> bool:
        d = {}
        for a, b, c in allowed:
            d.setdefault((a, b), []).append(c)
        level = {bottom}
        while len(next(iter(level))) > 1:
            nxt = set()
            for row in level:
                tops = []
                for a, b in zip(row, row[1:]):
                    cs = d.get((a, b))
                    if not cs:
                        break
                    tops.append(cs)
                else:
                    for combo in product(*tops):
                        nxt.add("".join(combo))
            if not nxt:
                return False
            level = nxt
        return True

for bottom, allowed, expected in examples:
    assert Solution().pyramidTransition(bottom, allowed) == expected
print("Cách 4 qua ví dụ đề")

Cách 4 qua ví dụ đề


**Độ phức tạp.** Mỗi tầng độ dài `m` có tối đa `6^m` hàng và mỗi hàng sinh tối đa `6^(m-1)` hàng con → cùng bậc với cách 2 trong tệ nhất (khoảng `10^7` khi `allowed` đủ 216 mẫu và đáy dài 6), nhưng BFS **luôn** đi hết các tầng nên chậm hơn khi bài "dễ" (nhiều mẫu). **Space O(6^m)** cho tập hàng ở tầng hiện tại.

## 8. TRACE từng bước

**Trace cách 2 (memo + product)** cho ví dụ 1, `bottom = "BCD"`, `allowed = ["BCC","CDE","CEA","FFF"]`:

| bước | `dfs(s)` | các cặp kề | danh sách đỉnh | tích Descartes | kết quả |
|---|---|---|---|---|---|
| 1 | `dfs("BCD")` | `(B,C)`, `(C,D)` | `[C]`, `[E]` | `("C","E")` → `"CE"` | gọi `dfs("CE")` |
| 2 | `dfs("CE")` | `(C,E)` | `[A]` | `("A",)` → `"A"` | gọi `dfs("A")` |
| 3 | `dfs("A")` | — | — | — | `len == 1` → **True** |

Ví dụ 2: `bottom = "AAAA"`, `allowed = ["AAB","AAC","BCD","BBE","DEF"]`. Mỗi cặp `(A,A)` có đỉnh `[B, C]`, ba cặp → `2×2×2 = 8` hàng tầng 3: `BBB, BBC, BCB, BCC, CBB, CBC, CCB, CCC`. Với mỗi hàng, tầng 2 cần mỗi cặp có mẫu: `BB→E`, `BC→D`, `CB→(không)`, `CC→(không)`. Hàng `BBB` → `"EE"` (cặp `EE` không có mẫu → thất bại); `BBC` → `"ED"` (cặp `ED` không có mẫu → thất bại); các hàng còn lại có cặp `CB` hoặc `CC` → thất bại ngay. Cả 8 thất bại → **False**.

In [19]:
def trace_dfs(bottom, allowed):
    d = build_table(allowed)
    memo = {}
    calls = [0]

    def dfs(s, depth):
        pad = "  " * depth
        if s in memo:
            print(f"{pad}dfs({s!r}) -> {memo[s]} (đã nhớ)")
            return memo[s]
        calls[0] += 1
        if len(s) == 1:
            print(f"{pad}dfs({s!r}) -> True (tới đỉnh)")
            memo[s] = True
            return True
        tops = []
        for a, b in pairwise(s):
            cs = d.get((a, b))
            if not cs:
                print(f"{pad}dfs({s!r}): cặp ({a},{b}) không có mẫu -> False")
                memo[s] = False
                return False
            tops.append(cs)
        rows = ["".join(x) for x in product(*tops)]
        print(f"{pad}dfs({s!r}): danh sách đỉnh {tops} -> các hàng kế tiếp {rows}")
        for r in rows:
            if dfs(r, depth + 1):
                memo[s] = True
                return True
        memo[s] = False
        print(f"{pad}dfs({s!r}) -> False (mọi hàng kế tiếp đều thất bại)")
        return False

    res = dfs(bottom, 0)
    print("kết quả:", res, "| số lần giải thật sự:", calls[0])
    return res

assert trace_dfs("BCD", ["BCC", "CDE", "CEA", "FFF"]) is True

dfs('BCD'): danh sách đỉnh [['C'], ['E']] -> các hàng kế tiếp ['CE']
  dfs('CE'): danh sách đỉnh [['A']] -> các hàng kế tiếp ['A']
    dfs('A') -> True (tới đỉnh)
kết quả: True | số lần giải thật sự: 3


In [20]:
assert trace_dfs("AAAA", ["AAB", "AAC", "BCD", "BBE", "DEF"]) is False

dfs('AAAA'): danh sách đỉnh [['B', 'C'], ['B', 'C'], ['B', 'C']] -> các hàng kế tiếp ['BBB', 'BBC', 'BCB', 'BCC', 'CBB', 'CBC', 'CCB', 'CCC']
  dfs('BBB'): danh sách đỉnh [['E'], ['E']] -> các hàng kế tiếp ['EE']
    dfs('EE'): cặp (E,E) không có mẫu -> False
  dfs('BBB') -> False (mọi hàng kế tiếp đều thất bại)
  dfs('BBC'): danh sách đỉnh [['E'], ['D']] -> các hàng kế tiếp ['ED']
    dfs('ED'): cặp (E,D) không có mẫu -> False
  dfs('BBC') -> False (mọi hàng kế tiếp đều thất bại)
  dfs('BCB'): cặp (C,B) không có mẫu -> False
  dfs('BCC'): cặp (C,C) không có mẫu -> False
  dfs('CBB'): cặp (C,B) không có mẫu -> False
  dfs('CBC'): cặp (C,B) không có mẫu -> False
  dfs('CCB'): cặp (C,C) không có mẫu -> False
  dfs('CCC'): cặp (C,C) không có mẫu -> False
dfs('AAAA') -> False (mọi hàng kế tiếp đều thất bại)
kết quả: False | số lần giải thật sự: 11


Chú ý: trong trace ví dụ 2, hàng `"EE"`, `"ED"`... mỗi hàng chỉ được **giải một lần**; nếu hai hàng tầng 3 cùng sinh ra một hàng tầng 2 thì lần sau ta thấy dòng "(đã nhớ)".

**Trace cách 3 (từng ký tự + bitmask)** cho ví dụ 1: `s = "BCD"`, `t = ""`: cặp `(B,C)` → mask chỉ có bit `C` → thử `C` → `t = "C"`; cặp `(C,D)` → mask chỉ có `E` → `t = "CE"`; `len(t) + 1 == len(s)` → `dfs("CE", "")` → cặp `(C,E)` → `A` → `t = "A"` → `dfs("A", "")` → `True`.

In [21]:
def trace_charwise(bottom, allowed):
    d = [[0] * 7 for _ in range(7)]
    for a, b, c in allowed:
        d[ord(a) - 65][ord(b) - 65] |= 1 << (ord(c) - 65)
    failed = set()

    def dfs(s, t, depth):
        pad = "  " * depth
        if len(s) == 1:
            print(f"{pad}s={s!r}: tới đỉnh -> True")
            return True
        if len(t) + 1 == len(s):
            print(f"{pad}s={s!r}, t={t!r}: hàng kế tiếp đủ -> chuyển lên hàng {t!r}")
            return dfs(t, "", depth)
        if (s, t) in failed:
            print(f"{pad}s={s!r}, t={t!r}: đã biết thất bại")
            return False
        a, b = s[len(t)], s[len(t) + 1]
        mask = d[ord(a) - 65][ord(b) - 65]
        cands = [chr(65 + i) for i in range(6) if mask >> i & 1]
        print(f"{pad}s={s!r}, t={t!r}: cặp ({a},{b}) mask={mask:06b} -> ứng viên {cands}")
        for c in cands:
            if dfs(s, t + c, depth + 1):
                return True
        failed.add((s, t))
        print(f"{pad}s={s!r}, t={t!r}: hết ứng viên -> thất bại")
        return False

    res = dfs(bottom, "", 0)
    print("kết quả:", res)
    return res

assert trace_charwise("BCD", ["BCC", "CDE", "CEA", "FFF"]) is True
print()
assert trace_charwise("AAAA", ["AAB", "AAC", "BCD", "BBE", "DEF"]) is False

s='BCD', t='': cặp (B,C) mask=000100 -> ứng viên ['C']
  s='BCD', t='C': cặp (C,D) mask=010000 -> ứng viên ['E']
    s='BCD', t='CE': hàng kế tiếp đủ -> chuyển lên hàng 'CE'
    s='CE', t='': cặp (C,E) mask=000001 -> ứng viên ['A']
      s='CE', t='A': hàng kế tiếp đủ -> chuyển lên hàng 'A'
      s='A': tới đỉnh -> True
kết quả: True

s='AAAA', t='': cặp (A,A) mask=000110 -> ứng viên ['B', 'C']
  s='AAAA', t='B': cặp (A,A) mask=000110 -> ứng viên ['B', 'C']
    s='AAAA', t='BB': cặp (A,A) mask=000110 -> ứng viên ['B', 'C']
      s='AAAA', t='BBB': hàng kế tiếp đủ -> chuyển lên hàng 'BBB'
      s='BBB', t='': cặp (B,B) mask=010000 -> ứng viên ['E']
        s='BBB', t='E': cặp (B,B) mask=010000 -> ứng viên ['E']
          s='BBB', t='EE': hàng kế tiếp đủ -> chuyển lên hàng 'EE'
          s='EE', t='': cặp (E,E) mask=000000 -> ứng viên []
          s='EE', t='': hết ứng viên -> thất bại
        s='BBB', t='E': hết ứng viên -> thất bại
      s='BBB', t='': hết ứng viên -> thất bại
      s=

Ở ví dụ 2, cách 3 cắt nhánh **sớm hơn** cách 2: chẳng hạn sau khi chọn `t = "C"` cho cặp đầu, cặp thứ hai có thể chọn `B` hoặc `C`, nhưng cặp `CB`/`CC` ở tầng kế tiếp sẽ khiến mọi nhánh kết thúc thất bại — và `failed` nhớ lại các trạng thái `(s, t)` đã hỏng.

**Trace cách 4 (BFS theo tầng)** cho ví dụ 2: `{"AAAA"}` → 8 hàng độ dài 3 → với mỗi hàng, ít nhất một cặp không có mẫu hoặc cặp cho hàng độ dài 2 mà cặp đó không có mẫu → tập độ dài 2 vẫn được dựng (`EE`, `ED`) nhưng tầng sau rỗng → `False`.

In [22]:
def trace_bfs(bottom, allowed):
    d = build_table(allowed)
    level = {bottom}
    print("tầng", len(bottom), ":", sorted(level))
    while len(next(iter(level))) > 1:
        nxt = set()
        for row in level:
            tops = []
            for a, b in pairwise(row):
                cs = d.get((a, b))
                if not cs:
                    break
                tops.append(cs)
            else:
                for combo in product(*tops):
                    nxt.add("".join(combo))
        print("tầng", len(next(iter(level))) - 1, ":", sorted(nxt))
        if not nxt:
            print("=> tập rỗng: False")
            return False
        level = nxt
    print("=> tới đỉnh: True")
    return True

assert trace_bfs("BCD", ["BCC", "CDE", "CEA", "FFF"]) is True
print()
assert trace_bfs("AAAA", ["AAB", "AAC", "BCD", "BBE", "DEF"]) is False

tầng 3 : ['BCD']
tầng 2 : ['CE']
tầng 1 : ['A']
=> tới đỉnh: True

tầng 4 : ['AAAA']
tầng 3 : ['BBB', 'BBC', 'BCB', 'BCC', 'CBB', 'CBC', 'CCB', 'CCC']
tầng 2 : ['ED', 'EE']
tầng 1 : []
=> tập rỗng: False


## 9. Code nộp LeetCode

Ta nộp **cách 3 (từng ký tự + bitmask + `failed`)**: cắt nhánh sớm, bộ nhớ nhỏ, không phụ thuộc thư viện nào ngoài hàm dựng sẵn; cách 2 cũng rất ổn và ngắn hơn nếu bạn thích `itertools`.

In [23]:
class Solution:
    def pyramidTransition(self, bottom: str, allowed: List[str]) -> bool:
        d = [[0] * 7 for _ in range(7)]
        for a, b, c in allowed:
            d[ord(a) - 65][ord(b) - 65] |= 1 << (ord(c) - 65)
        failed = set()

        def dfs(s: str, t: str) -> bool:
            if len(s) == 1:
                return True
            if len(t) + 1 == len(s):
                return dfs(t, "")
            key = (s, t)
            if key in failed:
                return False
            mask = d[ord(s[len(t)]) - 65][ord(s[len(t) + 1]) - 65]
            for i in range(6):
                if mask >> i & 1 and dfs(s, t + chr(65 + i)):
                    return True
            failed.add(key)
            return False

        return dfs(bottom, "")


sol = Solution()
for bottom, allowed, expected in examples:
    got = sol.pyramidTransition(bottom, allowed)
    print(f"{bottom!r} -> {got} (kỳ vọng {expected})")
    assert got == expected

'BCD' -> True (kỳ vọng True)
'AAAA' -> False (kỳ vọng False)
'AB' -> True (kỳ vọng True)
'AB' -> False (kỳ vọng False)


## 10. Vì sao đúng

**Mô hình.** Một kim tự tháp hợp lệ là dãy hàng `R_n = bottom, R_{n-1}, ..., R_1` với `|R_m| = m`, và với mọi `m ≥ 2`, mọi `c`: `R_m[c] + R_m[c+1] + R_{m-1}[c] ∈ allowed`. Các điều kiện ràng buộc **chỉ** giữa hai hàng liền nhau.

**Tính chất không nhớ (Markov).** Xây được tháp từ hàng `R_m` lên đỉnh ⇔ tồn tại `R_{m-1}` thỏa tam giác với `R_m` **và** xây được từ `R_{m-1}` lên đỉnh. Cái "xây được từ hàng `R` lên đỉnh" chỉ phụ thuộc `R` — nên ta có thể gọi nó `can(R)` và dùng bộ nhớ đệm đúng đắn (cách 2, 3, 4).

**Danh sách đỉnh theo cặp.** Với hàng `R` và hàng trên `U`, tam giác tại vị trí `c` hợp lệ ⇔ `U[c] ∈ d[(R[c], R[c+1])]`. Các vị trí `c` **độc lập** với nhau (mỗi `U[c]` chỉ liên quan tới cặp của chính nó) nên tập mọi `U` hợp lệ **đúng bằng tích Descartes** `d[(R[0],R[1])] × d[(R[1],R[2])] × ...`. Nếu có một cặp với danh sách rỗng, tích rỗng → không có `U` → `can(R) = False`. Cách 2 và 4 dùng chính tích Descartes; cách 3 dựng từng phần tử của tích bằng đệ quy theo vị trí `c` (cùng tập kết quả, chỉ khác thứ tự duyệt và cắt nhánh).

**Cách 3: tập thất bại đúng.** Trạng thái `(s, t)` nghĩa là "đang dựng hàng trên của `s`, đã có tiền tố `t`"; kết quả `dfs(s, t)` chỉ phụ thuộc `(s, t)` (và bảng cố định). Ta chỉ ghi `failed` khi **mọi** ứng viên đều thất bại nên việc trả `False` về sau là chính xác. Khi có một ứng viên thành công ta trả `True` cho tới hết đệ quy nên không cần lưu thành công.

**Cách 4: BFS đúng.** `level_m` = tập mọi hàng độ dài `m` đạt được từ `bottom` bằng các bước hợp lệ. Quy nạp: `level_{m-1}` = hợp mọi tích Descartes từ `level_m`. Tháp xây được ⇔ `level_1` khác rỗng (có đường tới đỉnh).

**Cách 1 đúng** theo định nghĩa (thử mọi hàng kế tiếp, không cắt gì).

**Dừng và phủ:** mỗi lần gọi giảm độ dài hàng đúng `1`, nên độ sâu đệ quy ≤ `n - 1`; số hàng hữu hạn nên thuật toán dừng.

## 11. Time / Space Complexity

Gọi `n = len(bottom) ≤ 6`, `s = 6`. Số hàng khác nhau độ dài `m` là `≤ 6^m`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | hàm mũ theo `n` (lặp lại nhiều) | O(độ sâu + hàng con) | để hiểu đề |
| 2. Memo + `product` | `≲ Σ 6^m · 6^(m-1)` tệ nhất (~10^7 ở m=5) | O(Σ 6^m) | gọn, dễ viết |
| 3. Từng ký tự + bitmask | cùng bậc, thường nhanh hơn nhờ cắt nhánh | O(trạng thái `(s,t)` thất bại) | **cách nộp** |
| 4. BFS theo tầng | cùng bậc; luôn đi hết các tầng | O(6^m) | không dừng sớm khi dễ |

Vì `n ≤ 6` và 6 chữ cái, mọi cách đều chạy trong thời gian chấp nhận được trên các test của đề.

## 12. Bộ test

Ta kiểm bằng ba lớp: **ví dụ + ca tính tay**, **trọng tài vét cạn độc lập** (`oracle` điền mọi ô) trên bảng chữ cái nhỏ, rồi **so các cách với nhau** ở cỡ tối đa. Khi trả `True` ta còn **dựng tháp cụ thể** và kiểm tra bằng `is_valid_pyramid` (mục 15.2).

In [24]:
all_ways = {
    "brute_force": brute_force,
    "memo_product": memo_product,
    "charwise_bitmask": charwise_bitmask,
    "bfs_levels": bfs_levels,
}

hand = [
    ("AB", [], False),                                   # không có mẫu nào
    ("AB", ["ABC"], True),
    ("AB", ["BAC"], False),                              # trái-phải quan trọng
    ("AA", ["AAA"], True),
    ("AAA", ["AAA"], True),                              # AA -> A, AA -> A, rồi AA -> A
    ("AAA", ["AAB"], False),                             # AA -> B, hàng "BB" không có mẫu
    ("AAA", ["AAB", "BBA"], True),                       # AAA -> BB -> A
    ("ABC", ["ABE", "ABD", "BCF", "DFA"], True),         # AB -> D (không phải E), BC -> F, DF -> A
    ("ABC", ["ABD"], False),                             # BC không có mẫu
    ("AAAAAA", ["AAA"], True),                           # đáy dài 6, mọi thứ là A
    ("ABCDEF", [], False),
    ("ABCDEF", ["ABA", "BCA", "CDA", "DEA", "EFA", "AAA"], True),   # tầng 5 toàn A, rồi AAA liên tiếp
]
tests = examples + hand
for bottom, allowed, expected in tests:
    for name, fn in all_ways.items():
        got = fn(bottom, allowed)
        assert got == expected, (name, bottom, allowed, got, expected)
print("OK:", len(tests), "ca cố định (ví dụ + tính tay), 4 cách khớp")

OK: 16 ca cố định (ví dụ + tính tay), 4 cách khớp


Kiểm tay ca `("AAA", ["AAB"])`: tầng 2 cần 2 khối, mỗi khối nằm trên cặp `(A,A)` → chỉ có đỉnh `B`, nên tầng 2 là `"BB"`. Tầng 1 cần mẫu `BB?` nhưng `allowed` chỉ có `AAB` → không có → `False`. Còn `("AAA", ["AAB","BBA"])`: `"BB"` rồi `BBA` cho đỉnh `A` → `True`.

Ca `("ABC", ["ABE","ABD","BCF","DFA"])` là ca **ghi chú cho lỗi "tham lam"** ở mục 13: cặp `AB` có hai đỉnh `E` và `D`; chọn `E` thì hàng `"EF"` không có mẫu (`EF?`), chọn `D` thì hàng `"DF"` có `DFA` → đúng `True`.

### Test ngẫu nhiên nhỏ so với trọng tài vét cạn

Ba cấu hình (seed cố định), bảng chữ cái và độ dài đáy khác nhau để `oracle` còn chạy kịp:

- chữ `AB`, đáy 2–6 (tối đa `2^15` cách điền);
- chữ `ABC`, đáy 2–4 (tối đa `3^6`);
- chữ `ABC`, đáy 5 (`3^10 = 59049`), chỉ vài ca.

`allowed` là một tập con ngẫu nhiên các mẫu (đúng ràng buộc: duy nhất, 3 ký tự).

In [25]:
def random_case(rng, letters, n, k_lo, k_hi):
    allp = [a + b + c for a in letters for b in letters for c in letters]
    k = rng.randint(min(k_lo, len(allp)), min(k_hi, len(allp)))
    allowed = rng.sample(allp, k)
    bottom = "".join(rng.choice(letters) for _ in range(n))
    return bottom, allowed

rng = random.Random(756)
n_true = n_false = 0
configs = [("AB", range(2, 7), 40, 0, 8), ("ABC", range(2, 5), 500, 0, 14), ("ABC", [5], 12, 4, 16)]
for letters, ns, count, klo, khi in configs:
    for _ in range(count):
        n = rng.choice(list(ns))
        bottom, allowed = random_case(rng, letters, n, klo, khi)
        want = oracle(bottom, allowed, letters)
        n_true += want
        n_false += not want
        for name, fn in all_ways.items():
            got = fn(bottom, allowed)
            assert got == want, (name, bottom, allowed, got, want)
print("OK: ngẫu nhiên nhỏ khớp oracle | True =", n_true, "| False =", n_false)
assert n_true > 50 and n_false > 50

OK: ngẫu nhiên nhỏ khớp oracle | True = 219 | False = 333


### Test cỡ đề (6 chữ cái, đáy tới 6)

Oracle không còn chạy nổi, nên so **cách 2, 3 và 4** với nhau (cách 1 chỉ cho các ca còn nhỏ). Khi kết quả là `True` ta kiểm chứng thêm bằng cách **dựng một tháp cụ thể** (`build_pyramid` ở mục 15.2) và kiểm tra `is_valid_pyramid`. Cách 4 (BFS) chậm khi `allowed` dày nên chỉ thử với số mẫu `≤ 60`.

In [26]:
def build_pyramid(bottom, allowed):
    """Trả về list các hàng từ đáy lên đỉnh nếu xây được, ngược lại None."""
    d = build_table(allowed)
    dead = set()

    def go(s):
        if len(s) == 1:
            return [s]
        if s in dead:
            return None
        tops = []
        for a, b in pairwise(s):
            cs = d.get((a, b))
            if not cs:
                dead.add(s)
                return None
            tops.append(cs)
        for combo in product(*tops):
            rest = go("".join(combo))
            if rest is not None:
                return [s] + rest
        dead.add(s)
        return None

    return go(bottom)

all216 = [a + b + c for a in "ABCDEF" for b in "ABCDEF" for c in "ABCDEF"]
rng = random.Random(7560)
stats = {True: 0, False: 0}
for t in range(400):
    n = rng.randint(2, 6)
    k = rng.choice([5, 15, 30, 60, 120, 216])
    allowed = rng.sample(all216, k)
    bottom = "".join(rng.choice("ABCDEF") for _ in range(n))
    ref = memo_product(bottom, allowed)
    assert charwise_bitmask(bottom, allowed) == ref, (bottom, allowed)
    if k <= 60:
        assert bfs_levels(bottom, allowed) == ref, (bottom, allowed)
    if n <= 4 and k <= 30:
        assert brute_force(bottom, allowed) == ref
    pyr = build_pyramid(bottom, allowed)
    assert (pyr is not None) == ref
    if pyr is not None:
        assert pyr[0] == bottom and is_valid_pyramid(pyr, allowed)
    stats[ref] += 1
print("OK: 400 ca cỡ đề | True =", stats[True], "| False =", stats[False])
assert stats[True] > 30 and stats[False] > 30

OK: 400 ca cỡ đề | True = 204 | False = 196


### Ca đặc biệt cỡ tối đa

- `allowed` đủ 216 mẫu và đáy 6: **luôn** `True` (mọi tam giác đều hợp lệ).
- `allowed` rỗng: **luôn** `False`.
- Đáy 6 chữ `A`, chỉ có mẫu `AAA`: `True`; chỉ có mẫu `AAB`: `False` (sau tầng 5 toàn `B`, không có mẫu `BB?`).

In [27]:
for bottom in ["ABCDEF", "AAAAAA", "FEDCBA", "ABABAB"]:
    for name, fn in [("memo_product", memo_product), ("charwise_bitmask", charwise_bitmask)]:
        assert fn(bottom, all216) is True, (name, bottom)
        assert fn(bottom, []) is False, (name, bottom)
for name, fn in all_ways.items():
    assert fn("AAAAAA", ["AAA"]) is True, name
    assert fn("AAAAAA", ["AAB"]) is False, name
print("OK: ca đặc biệt cỡ tối đa")

OK: ca đặc biệt cỡ tối đa


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Coi tam giác **không phân biệt trái–phải**

Mẫu `"ABC"` nghĩa là `A` **trái**, `B` **phải**. Nếu khóa bảng tra là một **tập** (`frozenset`) hoặc đã `sorted`, `("A","B")` và `("B","A")` bị coi là một, và đáy `"AB"` trúng mẫu `"BAC"`.

In [28]:
def wrong_unordered(bottom, allowed):
    d = {}
    for a, b, c in allowed:
        d.setdefault(frozenset((a, b)), []).append(c)      # SAI: bỏ thứ tự trái-phải

    @lru_cache(None)
    def dfs(s):
        if len(s) == 1:
            return True
        tops = []
        for a, b in pairwise(s):
            cs = d.get(frozenset((a, b)))
            if not cs:
                return False
            tops.append(cs)
        return any(dfs("".join(x)) for x in product(*tops))

    return dfs(bottom)

print("sai    : (AB, [BAC]) ->", wrong_unordered("AB", ["BAC"]), "| đúng:", memo_product("AB", ["BAC"]))
assert wrong_unordered("AB", ["BAC"]) is True
assert memo_product("AB", ["BAC"]) is False

sai    : (AB, [BAC]) -> True | đúng: False


**Sửa:** khóa là **tuple có thứ tự** `(a, b)`.

### Lỗi 2 — Tham lam: chỉ chọn ký tự đỉnh **đầu tiên** của mỗi cặp

Khi một cặp có nhiều ký tự đỉnh, lựa chọn đầu tiên có thể **dẫn vào ngõ cụt** trong khi lựa chọn khác thì thành công. Bài này cần **quay lui**, không tham lam.

In [29]:
def wrong_greedy(bottom, allowed):
    d = build_table(allowed)
    s = bottom
    while len(s) > 1:
        nxt = ""
        for a, b in pairwise(s):
            cs = d.get((a, b))
            if not cs:
                return False
            nxt += cs[0]                                   # SAI: luôn lấy ký tự đầu tiên
        s = nxt
    return True

allowed = ["ABE", "ABD", "BCF", "DFA"]
print("sai    : ('ABC', allowed) ->", wrong_greedy("ABC", allowed), "| đúng:", memo_product("ABC", allowed))
assert wrong_greedy("ABC", allowed) is False
assert memo_product("ABC", allowed) is True

sai    : ('ABC', allowed) -> False | đúng: True


**Vì sao sai?** Cặp `AB` có hai đỉnh `E`, `D` (theo thứ tự trong `allowed`). Tham lam chọn `E` → hàng `"EF"` rồi cặp `EF` không có mẫu → thất bại; nhưng chọn `D` → `"DF"` → `DFA` → thành công. **Sửa:** thử **mọi** lựa chọn (tích Descartes hoặc đệ quy từng ký tự).

### Lỗi 3 — Bỏ qua cặp không có mẫu (thay vì trả `False`)

Nếu một cặp không có mẫu mà ta chỉ `continue` (bỏ qua), hàng kế tiếp bị **ngắn hơn** đáng lẽ và thuật toán "xây" được một tháp không hợp lệ.

In [30]:
def wrong_skip_missing(bottom, allowed):
    d = build_table(allowed)

    @lru_cache(None)
    def dfs(s):
        if len(s) == 1:
            return True
        tops = []
        for a, b in pairwise(s):
            cs = d.get((a, b))
            if cs:                                         # SAI: cặp không có mẫu bị bỏ qua
                tops.append(cs)
        return any(dfs("".join(x)) for x in product(*tops))

    return dfs(bottom)

print("sai    : ('ABC', ['ABD']) ->", wrong_skip_missing("ABC", ["ABD"]), "| đúng:", memo_product("ABC", ["ABD"]))
assert wrong_skip_missing("ABC", ["ABD"]) is True
assert memo_product("ABC", ["ABD"]) is False

sai    : ('ABC', ['ABD']) -> True | đúng: False


**Sửa:** gặp cặp không có mẫu → `return False` ngay (hoặc `tops` rỗng → tích rỗng).

### Lỗi 4 — Bộ nhớ đệm **toàn cục** khi bảng `allowed` đổi giữa các lần gọi

Nếu đặt `@lru_cache` trên hàm ở mức module và hàm đọc bảng `D` **toàn cục**, kết quả của lần gọi trước còn nằm trong cache và bị dùng nhầm cho lần gọi sau (khác `allowed`). LeetCode chạy nhiều test trong cùng tiến trình — rất nguy hiểm.

In [31]:
D = {}

@lru_cache(None)
def stale_dfs(s):                                          # SAI: cache sống lâu hơn bảng D
    if len(s) == 1:
        return True
    tops = []
    for a, b in pairwise(s):
        cs = D.get((a, b))
        if not cs:
            return False
        tops.append(cs)
    return any(stale_dfs("".join(x)) for x in product(*tops))

def wrong_global_cache(bottom, allowed):
    D.clear()
    D.update(build_table(allowed))
    return stale_dfs(bottom)

print("lần 1 (có mẫu) :", wrong_global_cache("BCD", ["BCC", "CDE", "CEA"]))
print("lần 2 (không mẫu):", wrong_global_cache("BCD", []), "<- đúng phải là False")
assert wrong_global_cache("BCD", []) is True
assert memo_product("BCD", []) is False

lần 1 (có mẫu) : True
lần 2 (không mẫu): True <- đúng phải là False


**Sửa:** tạo `d` và hàm có cache **bên trong** `pyramidTransition` (như mọi cách nộp trong notebook này) để mỗi lần gọi có bộ nhớ riêng.

### Lỗi 5 — Sai chỉ số khi tách cặp kề (`s[i:i+2]` với `i` tới hết)

`for i in range(len(s))` rồi `s[i:i+2]` cho cặp cuối chỉ có **một** ký tự (cắt lát không báo lỗi!). Bảng tra không có khóa như vậy → thuật toán luôn thất bại, hoặc nếu bạn "chữa" bằng `.get(..., [])` thì tích rỗng.

In [32]:
def wrong_pair_index(bottom, allowed):
    d = {}
    for a, b, c in allowed:
        d.setdefault(a + b, []).append(c)

    @lru_cache(None)
    def dfs(s):
        if len(s) == 1:
            return True
        tops = []
        for i in range(len(s)):                            # SAI: phải là len(s) - 1
            cs = d.get(s[i:i + 2])
            if not cs:
                return False
            tops.append(cs)
        return any(dfs("".join(x)) for x in product(*tops))

    return dfs(bottom)

print("sai    : ('AB', ['ABC']) ->", wrong_pair_index("AB", ["ABC"]), "| đúng:", memo_product("AB", ["ABC"]))
assert wrong_pair_index("AB", ["ABC"]) is False
assert memo_product("AB", ["ABC"]) is True

sai    : ('AB', ['ABC']) -> False | đúng: True


**Sửa:** dùng `zip(s, s[1:])` / `pairwise(s)` hoặc `range(len(s) - 1)`.

## 14. Mẹo & nhận dạng pattern

- **"Xây từng tầng, mỗi tầng phụ thuộc tầng trước"** + **không gian nhỏ** → quay lui với **memo theo trạng thái = cả tầng** (chuỗi hashable).
- **"Mỗi vị trí có nhiều lựa chọn độc lập"** → tích Descartes (`itertools.product`) liệt kê mọi tổ hợp; hoặc **dựng từng phần tử** bằng DFS để **cắt nhánh sớm**.
- **Dạng bảng tra "cặp → tập"** → `dict` với khóa tuple, hoặc mảng hai chiều + **bitmask** khi bảng chữ cái nhỏ (≤ 6–26).
- **Bài có "trái–phải" khác nhau**: khóa phải giữ **thứ tự**; đừng sort/set.
- **Kiểm thử bài quay lui**: viết `oracle` vét cạn rất ngây thơ (điền mọi ô) cho cỡ nhỏ, và một hàm **kiểm tra lời giải cụ thể** (`is_valid_pyramid`) để chứng thực mỗi khi trả `True`.
- **Cache an toàn**: đặt hàm có cache **trong** hàm chính để không dính trạng thái giữa các test.
- Họ hàng: *Word Ladder* (127), *Generate Parentheses* (22), *N-Queens* (51) cho DFS/quay lui; *Number of Ways to Form a Target String* cho DP có trạng thái nhỏ.

## 15. Biến thể

### 15.1. **Đếm** số kim tự tháp hợp lệ

Thay `any` bằng **tổng**: số tháp hoàn chỉnh = tổng, trên mọi hàng kế tiếp `U`, của số tháp xây tiếp từ `U`. Nhớ bằng `lru_cache`. Kiểm chứng bằng cách đếm trực tiếp mọi cách điền (cỡ nhỏ).

In [33]:
def count_pyramids(bottom, allowed):
    d = build_table(allowed)

    @lru_cache(None)
    def cnt(s):
        if len(s) == 1:
            return 1
        tops = []
        for a, b in pairwise(s):
            cs = d.get((a, b))
            if not cs:
                return 0
            tops.append(cs)
        return sum(cnt("".join(x)) for x in product(*tops))

    return cnt(bottom)

def count_oracle(bottom, allowed, letters):
    n = len(bottom)
    S = set(allowed)
    total = n * (n - 1) // 2
    ways = 0
    for assign in product(letters, repeat=total):
        rows = [bottom]
        idx = 0
        for length in range(n - 1, 0, -1):
            rows.append("".join(assign[idx:idx + length]))
            idx += length
        if all(rows[r][c] + rows[r][c + 1] + rows[r + 1][c] in S
               for r in range(n - 1) for c in range(len(rows[r + 1]))):
            ways += 1
    return ways

print(count_pyramids("AAA", ["AAA", "AAB", "BBA", "ABA"]))       # số cách
assert count_pyramids("BCD", ["BCC", "CDE", "CEA", "FFF"]) == 1
assert count_pyramids("AAAA", ["AAB", "AAC", "BCD", "BBE", "DEF"]) == 0
rng = random.Random(151)
for _ in range(150):
    n = rng.randint(2, 4)
    bottom, allowed = random_case(rng, "AB", n, 0, 8)
    assert count_pyramids(bottom, allowed) == count_oracle(bottom, allowed, "AB")
print("count_pyramids khớp đếm vét cạn")

4
count_pyramids khớp đếm vét cạn


### 15.2. Dựng **một** tháp cụ thể

Hàm `build_pyramid` ở mục 12 trả list hàng từ đáy lên đỉnh (hoặc `None`). Hãy in thử để thấy tháp ở ví dụ 1 và dùng `is_valid_pyramid` kiểm.

In [34]:
pyr = build_pyramid("BCD", ["BCC", "CDE", "CEA", "FFF"])
print(pyr)
show(pyr)
assert pyr == ["BCD", "CE", "A"] and is_valid_pyramid(pyr, ["BCC", "CDE", "CEA", "FFF"])
assert build_pyramid("AAAA", ["AAB", "AAC", "BCD", "BBE", "DEF"]) is None

['BCD', 'CE', 'A']
  A  
 C E 
B C D


### 15.3. Những chữ cái nào có thể nằm trên **đỉnh**?

Thay vì `True/False`, trả về tập mọi ký tự có thể ở đỉnh. Gợi ý: hợp (union) kết quả của mọi hàng kế tiếp. Đây cũng là bài tập 5 ở mục 17.

In [35]:
def possible_tops(bottom, allowed):
    d = build_table(allowed)

    @lru_cache(None)
    def tops_of(s):
        if len(s) == 1:
            return frozenset(s)
        tops = []
        for a, b in pairwise(s):
            cs = d.get((a, b))
            if not cs:
                return frozenset()
            tops.append(cs)
        res = set()
        for x in product(*tops):
            res |= tops_of("".join(x))
        return frozenset(res)

    return "".join(sorted(tops_of(bottom)))

print(possible_tops("BCD", ["BCC", "CDE", "CEA", "FFF"]))        # 'A'
print(possible_tops("AB", ["ABC", "ABD"]))                        # 'CD'
assert possible_tops("BCD", ["BCC", "CDE", "CEA", "FFF"]) == "A"
assert possible_tops("AB", ["ABC", "ABD"]) == "CD"
assert possible_tops("AAAA", ["AAB", "AAC", "BCD", "BBE", "DEF"]) == ""

A
CD


### 15.4. Bài liên quan

- **Pyramid Transition Matrix II** (nếu có) hoặc các bài quay lui theo tầng; **22. Generate Parentheses**, **39. Combination Sum** — DFS sinh tổ hợp.
- **127. Word Ladder** — BFS theo tầng trên tập chuỗi (giống cách 4).
- **212. Word Search II**, **79. Word Search** — DFS có cắt nhánh.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi hàng kế tiếp, không nhớ | hàm mũ, lặp nhiều | O(độ sâu + hàng con) | chỉ để hiểu; làm đối chiếu |
| 2. Memo + `product` | nhớ mỗi hàng; hàng kế tiếp = tích Descartes | `≲ Σ 6^m · 6^(m-1)` | O(Σ 6^m) | gọn, dễ viết |
| 3. Từng ký tự + bitmask | dựng hàng kế tiếp từng ký tự, cắt nhánh sớm, nhớ thất bại | cùng bậc, thường nhanh hơn | nhỏ | **nên nộp** |
| 4. BFS theo tầng | gom mọi hàng cùng độ dài thành tập | cùng bậc; không dừng sớm | O(6^m) | thích vòng lặp thay đệ quy; `allowed` thưa |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng — **không được gọi** trước khi bạn viết xong. Đáp án ở mục 19.

### Bài 1 (đoán output)

Không chạy code, hãy đoán `result` (một tuple gồm 3 phần tử) rồi điền vào `my_guess_1`.

In [36]:
allowed = ["ABC", "ABD", "BAE"]
d = build_table(allowed)
result = (d.get(("A", "B")), d.get(("B", "A")), d.get(("C", "C")))
# print(result)   # <- bỏ dấu # sau khi đã đoán xong để kiểm tra

my_guess_1 = None    # TODO: điền tuple dạng (..., ..., ...)

### Bài 2 (tính tay)

Với `bottom = "ABC"`, `allowed = ["ABD", "ABE", "BCF", "DFA"]`: có xây được tới đỉnh không? Hãy liệt kê tay các hàng tầng 2 có thể có và kiểm từng hàng. Điền `True`/`False` vào `my_answer_2`.

In [37]:
my_answer_2 = None    # TODO: True hoặc False

### Bài 3 (điền TODO)

Hoàn thành hàm DFS có memo: thay `...` bằng biểu thức/câu lệnh đúng.

In [38]:
def exercise3(bottom, allowed):
    d = build_table(allowed)

    @lru_cache(None)
    def dfs(s):
        if ...:                                  # TODO: điều kiện dừng (tới đỉnh)
            return True
        tops = []
        for a, b in pairwise(s):
            cs = d.get(...)                      # TODO: tra bảng với khóa nào?
            if not cs:
                return ...                       # TODO: cặp không có mẫu -> kết quả gì?
            tops.append(cs)
        return any(dfs(...) for nxt in product(*tops))   # TODO: gọi đệ quy với chuỗi hàng kế tiếp

    return dfs(bottom)

### Bài 4 (tự viết lại)

Tự viết `exercise4(bottom, allowed)` giải bài bằng **BFS theo tầng với tập các hàng** (cách 4), không nhìn lại mục 7.3. Trả `True`/`False`.

In [39]:
def exercise4(bottom, allowed):
    # TODO: viết lời giải
    pass

### Bài 5 (biến thể / mini challenge)

Viết `exercise5(bottom, allowed)` trả về **chuỗi các chữ cái (sắp xếp tăng dần, không lặp)** có thể nằm ở **đỉnh** của một kim tự tháp hợp lệ; nếu không xây được thì trả `""` (xem mục 15.3). Ví dụ `exercise5("AB", ["ABC", "ABD"])` → `"CD"`.

In [40]:
def exercise5(bottom, allowed):
    # TODO: viết lời giải
    pass

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `build_table` gom các mẫu theo **hai ký tự đầu**.
- Mức 2: `("A","B")` có hai mẫu: `ABC`, `ABD`; `("B","A")` có mẫu `BAE`; `("C","C")` không có.
- Mức 3: `d.get` trả list (theo thứ tự trong `allowed`) hoặc `None` nếu thiếu khóa (không phải `[]` vì không truyền mặc định!).

**Bài 2**
- Mức 1: cặp `(A,B)` có đỉnh `D` hoặc `E`; cặp `(B,C)` có đỉnh `F`.
- Mức 2: hai hàng tầng 2: `"DF"` và `"EF"`. Kiểm từng hàng với `allowed`.
- Mức 3: `DF` có mẫu `DFA` → tới đỉnh; `EF` không có mẫu → thất bại. Có một hàng thành công → `True`.

**Bài 3**
- Mức 1: nhìn lại mục 7.1.
- Mức 2: dừng khi `len(s) == 1`; khóa tra bảng là `(a, b)`.
- Mức 3: `len(s) == 1`; `d.get((a, b))`; `return False`; `dfs("".join(nxt))`.

**Bài 4**
- Mức 1: `level = {bottom}`; lặp cho tới khi độ dài hàng còn 1.
- Mức 2: với mỗi hàng, nếu mọi cặp có mẫu thì thêm mọi tích Descartes vào tập mới; tập mới rỗng → `False`.
- Mức 3: dùng `for ... else` hoặc cờ để bỏ hàng có cặp thiếu mẫu.

**Bài 5**
- Mức 1: khi `len(s) == 1` tập đỉnh là `{s}`.
- Mức 2: với mỗi hàng kế tiếp, hợp (union) tập đỉnh trả về.
- Mức 3: cuối cùng `"".join(sorted(tập))`; không xây được thì tập rỗng cho `""`.

## 19. Đáp án tham khảo

Chạy các cell sau để kiểm tra bằng `assert`.

In [41]:
# Bài 1
allowed = ["ABC", "ABD", "BAE"]
d = build_table(allowed)
answer_1 = (d.get(("A", "B")), d.get(("B", "A")), d.get(("C", "C")))
print("Bài 1:", answer_1)
assert answer_1 == (["C", "D"], ["E"], None)

# Bài 2
answer_2 = memo_product("ABC", ["ABD", "ABE", "BCF", "DFA"])
print("Bài 2:", answer_2)
assert answer_2 is True
assert oracle("ABC", ["ABD", "ABE", "BCF", "DFA"], "ABCDEF") is True

Bài 1: (['C', 'D'], ['E'], None)
Bài 2: True


In [42]:
# Bài 3
def exercise3_solution(bottom, allowed):
    d = build_table(allowed)

    @lru_cache(None)
    def dfs(s):
        if len(s) == 1:
            return True
        tops = []
        for a, b in pairwise(s):
            cs = d.get((a, b))
            if not cs:
                return False
            tops.append(cs)
        return any(dfs("".join(nxt)) for nxt in product(*tops))

    return dfs(bottom)

# Bài 4
def exercise4_solution(bottom, allowed):
    d = build_table(allowed)
    level = {bottom}
    while len(next(iter(level))) > 1:
        nxt = set()
        for row in level:
            tops = []
            for a, b in pairwise(row):
                cs = d.get((a, b))
                if not cs:
                    break
                tops.append(cs)
            else:
                for combo in product(*tops):
                    nxt.add("".join(combo))
        if not nxt:
            return False
        level = nxt
    return True

# Bài 5
def exercise5_solution(bottom, allowed):
    d = build_table(allowed)

    @lru_cache(None)
    def tops_of(s):
        if len(s) == 1:
            return frozenset(s)
        tops = []
        for a, b in pairwise(s):
            cs = d.get((a, b))
            if not cs:
                return frozenset()
            tops.append(cs)
        res = set()
        for combo in product(*tops):
            res |= tops_of("".join(combo))
        return frozenset(res)

    return "".join(sorted(tops_of(bottom)))

for bottom, allowed, expected in examples:
    assert exercise3_solution(bottom, allowed) == expected
    assert exercise4_solution(bottom, allowed) == expected
assert exercise5_solution("AB", ["ABC", "ABD"]) == "CD"
assert exercise5_solution("BCD", ["BCC", "CDE", "CEA", "FFF"]) == "A"
assert exercise5_solution("AB", []) == ""
print("Đáp án bài 3, 4, 5 đúng trên ví dụ")

rng = random.Random(19)
for _ in range(400):
    n = rng.randint(2, 4)
    bottom, allowed = random_case(rng, "ABC", n, 0, 14)
    want = oracle(bottom, allowed, "ABC")
    assert exercise3_solution(bottom, allowed) == want
    assert exercise4_solution(bottom, allowed) == want
    assert (exercise5_solution(bottom, allowed) != "") == want
print("Đối chiếu ngẫu nhiên OK")

Đáp án bài 3, 4, 5 đúng trên ví dụ


Đối chiếu ngẫu nhiên OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (từng ký tự + bitmask + tập thất bại):**

```python
class Solution:
    def pyramidTransition(self, bottom: str, allowed: List[str]) -> bool:
        d = [[0] * 7 for _ in range(7)]               # d[a][b] = bitmask các đỉnh cho phép
        for a, b, c in allowed:
            d[ord(a) - 65][ord(b) - 65] |= 1 << (ord(c) - 65)
        failed = set()

        def dfs(s: str, t: str) -> bool:              # s: hàng hiện tại, t: hàng kế tiếp đang dựng
            if len(s) == 1:
                return True                           # tới đỉnh
            if len(t) + 1 == len(s):
                return dfs(t, "")                     # hàng kế tiếp đủ -> lên tầng
            key = (s, t)
            if key in failed:
                return False
            mask = d[ord(s[len(t)]) - 65][ord(s[len(t) + 1]) - 65]
            for i in range(6):
                if mask >> i & 1 and dfs(s, t + chr(65 + i)):
                    return True
            failed.add(key)
            return False

        return dfs(bottom, "")
```

**Checklist trước khi submit:**

- [ ] Khóa tra bảng là `(trái, phải)` **có thứ tự**: `"ABC"` ≠ `"BAC"`.
- [ ] Thử **mọi** ký tự đỉnh của mỗi cặp (quay lui), không tham lam.
- [ ] Cặp **không có mẫu** → thất bại (`False`), không bỏ qua.
- [ ] Trạng thái nhớ là **cả hàng** (chuỗi) hoặc `(s, t)`; bộ nhớ đệm đặt **bên trong** hàm chính.
- [ ] Tách cặp bằng `zip(s, s[1:])`/`pairwise`, tránh `s[i:i+2]` đến hết chuỗi.
- [ ] Cơ sở `len(s) == 1` trả `True`; độ sâu đệ quy nhỏ (≤ `n`).
- [ ] Đã thử: `allowed` rỗng, `AB` với `ABC` và `BAC`, `AAAAAA` với `AAA`, ca có 2 ứng viên đỉnh (tham lam sai).
- [ ] Đã đối chiếu với `oracle` vét cạn trên bảng chữ cái nhỏ và kiểm tháp bằng `is_valid_pyramid`.